# 뮤직비디오 장르·가사 크롤링 

곡명과 아티스트명으로 음원 정보 사이트에서 곡을 검색해 장르(`site_genre`)와 가사(`site_lyrics`)를 크롤링하는 코드입니다. Selenium으로 음원 정보 사이트 웹사이트를 직접 조작해 검색 → 첫 검색결과 클릭 → 장르/가사 텍스트 추출 순으로 동작합니다.

**구성**
1. 메인 크롤링 — 전체 곡 목록을 대상으로 장르·가사 수집
2. 미수집 재크롤링 — 장르 또는 가사가 비어있는 곡만 골라 다시 시도

**참고사항**
- Selenium + Chrome WebDriver를 사용하므로 로컬/Colab 등 브라우저 실행이 가능한 환경이 필요합니다.
- 음원 정보 사이트 페이지 구조가 바뀌면 XPath가 깨질 수 있습니다.
- 원본 곡 목록과 크롤링 결과 파일은 저작권이 있는 곡 정보(가사 포함)를 담고 있어 이 저장소에는 포함하지 않았습니다.


## 1. 메인 크롤링

Selenium으로 음원 정보 사이트에서 곡명+아티스트를 검색해 첫 검색결과의 장르와 가사를 가져옵니다. 100곡마다 중간저장하며, 전체 곡 목록에 `site_genre`/`site_lyrics` 컬럼을 채운 뒤 CSV로 저장합니다.


In [ ]:
!pip install selenium webdriver-manager -q
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import re

# =========================
# 🔹 크롬 설정
# =========================
options = Options()
# options.add_argument("--headless")  # 화면 보고 싶으면 주석처리
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# =========================
# 🔹 장르 + 가사 수집 함수
# =========================
def get_genre_lyrics(song, artist):
    try:
        driver.get("https://target_site.com/")
        time.sleep(2)
        search_box = driver.find_element(By.ID, "sc-fd")
        search_box.clear()
        search_box.send_keys(f"{song} {artist}")
        search_box.send_keys(Keys.ENTER)
        time.sleep(2)
        
        # 첫 곡 클릭
        first_song_xpath = '//*[@id="body-content"]/div[3]/div[2]/div/table/tbody/tr[1]/td[4]/a'
        driver.find_element(By.XPATH, first_song_xpath).click()
        time.sleep(2)
        
        # 장르 추출
        genre_xpath = '//*[@id="body-content"]/div[2]/div[2]/ul/li[3]/span[2]'
        genre = driver.find_element(By.XPATH, genre_xpath).text
        
        # 가사 추출
        try:
            lyrics_xpath = '//*[@id="pLyrics"]/p'
            lyrics_element = driver.find_element(By.XPATH, lyrics_xpath)
            lyrics_html = lyrics_element.get_attribute('innerHTML')
            lyrics = lyrics_html.replace('<br>', '\n').strip()
            lyrics = re.sub(r'<[^>]+>', '', lyrics)
        except:
            lyrics = None
        
        return genre, lyrics
    except:
        return None, None

# =========================
# 🔹 데이터 불러오기
# =========================
df = pd.read_csv(r"C:\Users\sy-77\Downloads\combined_master_list(전체크롤링용).csv", encoding='utf-8-sig')

# =========================
# 🔹 artist_kr, artist_en 추가
# =========================
def extract_artists(artists_str):
    try:
        parts = str(artists_str).split('|')
        artist_kr = parts[1] if len(parts) > 1 else ""
        artist_en = parts[2] if len(parts) > 2 else ""
        return artist_kr, artist_en
    except:
        return "", ""

df[['artist_kr', 'artist_en']] = df['artists'].apply(lambda x: pd.Series(extract_artists(x)))

if "site_genre" not in df.columns:
    df["site_genre"] = None

if "site_lyrics" not in df.columns:
    df["site_lyrics"] = None

print(f"🎵 총 {len(df)}곡 장르 수집 시작\n")

# =========================
# 🔹 전체 반복
# =========================
for i in range(len(df)):
    # 이미 수집된 곡은 스킵 가능
    if pd.notna(df.loc[i, "site_genre"]) and pd.notna(df.loc[i, "site_lyrics"]):
        continue
    
    song = df.loc[i, "songName"]
    artist = df.loc[i, "artist_kr"]
    
    print(f"🔎 {i+1}/{len(df)}: {song} - {artist}")
    
    genre, lyrics = get_genre_lyrics(song, artist)
    
    df.loc[i, "site_genre"] = genre
    df.loc[i, "site_lyrics"] = lyrics
    
    print(f"   🎼 장르: {genre}")
    print(f"   📝 가사: {lyrics[:30] if lyrics else None}...")
    
    # 🔥 100곡마다 저장 (중간저장)
    if i % 100 == 0:
        df.to_csv(r"C:\Users\sy-77\Downloads\combined_master_list_with_genre.csv", index=False, encoding='utf-8-sig')
        print("💾 중간 저장 완료")
    
    # 🔥 속도 조절 (차단 방지)
    time.sleep(2)

driver.quit()

# 최종 저장
df.to_csv(r"C:\Users\sy-77\Downloads\combined_master_list_with_genre.csv", index=False, encoding='utf-8-sig')
print("\n===== 🎉 전체 완료 =====")
print("파일 저장: C:\\Users\\sy-77\\Downloads\\combined_master_list_with_genre.csv")

## 2. 미수집 재크롤링

기존 결과에서 `site_genre` 또는 `site_lyrics`가 비어있는 행만 골라내 다시 크롤링합니다. 50곡마다 중간저장하며 예상 완료 시각을 함께 출력합니다.


In [ ]:
# 1. 필수 라이브러리 설치
!pip install selenium webdriver-manager -q

import time
import pandas as pd
import re
import os
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# 🔹 1. 크롬 드라이버 설정
# ==========================================
options = Options()
# options.add_argument("--headless") # 과정을 보려면 주석처리
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================
# 🔹 2. 장르·가사 수집 함수
# ==========================================
def get_genre_lyrics(song, artist):
    try:
        driver.get("https://target_site.com/")
        time.sleep(1.2)
        
        # 검색어 입력
        search_box = driver.find_element(By.ID, "sc-fd")
        search_box.clear()
        search_box.send_keys(f"{song} {artist}")
        search_box.send_keys(Keys.ENTER)
        time.sleep(1.2)
        
        # 첫 번째 검색 결과 클릭
        try:
            first_song_xpath = '//*[@id="body-content"]/div[3]/div[2]/div/table/tbody/tr[1]/td[4]/a'
            driver.find_element(By.XPATH, first_song_xpath).click()
            time.sleep(1.2)
        except:
            return None, None # 검색 결과 없음
        
        # 장르 추출
        try:
            genre_xpath = '//*[@id="body-content"]/div[2]/div[2]/ul/li[3]/span[2]'
            genre = driver.find_element(By.XPATH, genre_xpath).text
        except:
            genre = "None"
            
        # 가사 추출
        try:
            lyrics_xpath = '//*[@id="pLyrics"]/p'
            lyrics_element = driver.find_element(By.XPATH, lyrics_xpath)
            lyrics_html = lyrics_element.get_attribute('innerHTML')
            # <br> 태그를 줄바꿈으로 변경 및 HTML 태그 제거
            lyrics = lyrics_html.replace('<br>', '\n').strip()
            lyrics = re.sub(r'<[^>]+>', '', lyrics)
        except:
            lyrics = None
        
        return genre, lyrics
    except Exception as e:
        return None, None

# ==========================================
# 🔹 3. 데이터 로드 및 타겟 인덱스 설정
# ==========================================
input_path = r"C:\Users\sy-77\Downloads\merged_df_이름정리.csv"
output_path = r"C:\Users\sy-77\Downloads\merged_df_가사_장르_재추출.csv"

# 데이터 불러오기
df = pd.read_csv(input_path, encoding='utf-8-sig')

# 미수집 대상 선정 (장르 또는 가사가 없는 경우)
target_indices = df[df['site_genre'].isna() | df['site_lyrics'].isna()].index
total_targets = len(target_indices)

print("="*70)
print(f"📂 입력: {input_path}")
print(f"🎯 미수집 타겟: {total_targets}개 (전체 {len(df)}개 중)")
print(f"⏰ 시작 시간: {datetime.now().strftime('%H:%M:%S')}")
print("="*70)

# ==========================================
# 🔹 4. 메인 루프 (실시간 출력 강화)
# ==========================================
processed = 0
success_count = 0
start_time = time.time()

for idx in target_indices:
    processed += 1
    song = df.loc[idx, "songName"]
    # 괄호와 숫자가 제거된 깨끗한 국문 이름 사용
    artist = df.loc[idx, "kor_name"] if pd.notna(df.loc[idx, "kor_name"]) else df.loc[idx, "eng_name"]
    
    # 크롤링 실행
    genre, lyrics = get_genre_lyrics(song, artist)
    
    # 데이터 업데이트
    if genre:
        df.loc[idx, "site_genre"] = genre
    if lyrics:
        df.loc[idx, "site_lyrics"] = lyrics
        success_count += 1

    # 🖥️ 실시간 모니터링 출력
    print(f"🔎 {processed}/{total_targets}: {song} - {artist}")
    print(f"   🎼 장르: {genre if genre else 'None'}")
    
    if lyrics:
        preview = lyrics[:60].replace('\n', ' ')
        print(f"   📝 가사: {preview}...")
    else:
        print(f"   📝 가사: None")
    print("-" * 50)

    # 50곡마다 중간 저장 및 예상 완료 시간 출력
    if processed % 50 == 0:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        elapsed = time.time() - start_time
        avg_time = elapsed / processed
        eta_sec = avg_time * (total_targets - processed)
        eta_time = (datetime.now() + timedelta(seconds=eta_sec)).strftime('%H:%M:%S')
        
        print(f"\n💾 중간 저장 완료! | 예상 완료: {eta_time} ({int(eta_sec/60)}분 남음)\n")

    time.sleep(1.2) # 차단 방지 매너 타임

# ==========================================
# 🔹 5. 최종 완료 및 저장
# ==========================================
driver.quit()
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n" + "="*70)
print(f"✅ 모든 수집 작업 완료!")
print(f"📊 최종 성공 곡 수: {success_count}개")
print(f"📁 파일 저장 위치: {output_path}")
print("="*70)